In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])

import nibabel as nib
import numpy as np
import pandas as pd
import os

# 경로 설정
t1_dir    = r"D:\ADNI\raw_nifti"
clean_dir = r"D:\ADNI\clean_masks_hippo"
meta_path = r"D:\ADNI\metadata\adni_merged_icv.csv"
out_dir   = r"D:\ADNI\feature"
os.makedirs(out_dir, exist_ok=True)

meta = None
if os.path.exists(meta_path):
    meta = pd.read_csv(meta_path)
    print("Loaded meta:", meta_path)
else:
    print("[경고] 메타데이터 파일 없음:", meta_path)

rows = []

for fname in os.listdir(clean_dir):
    if not fname.endswith("_hipp_clean.nii.gz"):
        continue

    mask_path = os.path.join(clean_dir, fname)
    print(f"\n[ Feature for: {mask_path} ]")

    base = fname.replace("_hipp_clean.nii.gz", "")
    t1_fname = base + ".nii.gz"
    t1_path = os.path.join(t1_dir, t1_fname)

    if not os.path.exists(t1_path):
        print(f"[경고] T1 파일 없음, 스킵: {t1_path}")
        continue

    # 1) 마스크 로드 & voxel volume
    mask_img = nib.load(mask_path)
    mask = mask_img.get_fdata()
    voxel_sizes = mask_img.header.get_zooms()[:3]
    voxel_volume = voxel_sizes[0] * voxel_sizes[1] * voxel_sizes[2]

    print("voxel sizes (mm):", voxel_sizes)
    print("voxel volume (mm^3):", voxel_volume)

    # 2) 좌/우 해마 분리
    left_mask  = (mask == 1)
    right_mask = (mask == 2)

    left_voxels  = int(left_mask.sum())
    right_voxels = int(right_mask.sum())

    left_vol_mm3  = left_voxels  * voxel_volume
    right_vol_mm3 = right_voxels * voxel_volume
    total_vol_mm3 = left_vol_mm3 + right_vol_mm3

    if total_vol_mm3 > 0:
        asymmetry_index = (right_vol_mm3 - left_vol_mm3) / total_vol_mm3
    else:
        asymmetry_index = np.nan

    print("L voxels:", left_voxels, "R voxels:", right_voxels)
    print("L vol:", left_vol_mm3, "R vol:", right_vol_mm3)

    # 3) subject_id / scan_id
    parts = base.split("_")
    subject_id = "_".join(parts[0:3])
    scan_id    = parts[3] if len(parts) > 3 else ""

    # 4) 메타에서 ICV / DX
    icv = np.nan
    dx = None

    if meta is not None and "PTID" in meta.columns:
        cand = meta[meta["PTID"] == subject_id]
        if not cand.empty:
            row = cand.iloc[0]
            if "ICV" in row.index:
                icv = row["ICV"]
            if "DX" in row.index:
                dx = row["DX"]

    if np.isfinite(icv) and icv > 0:
        left_norm  = left_vol_mm3  / icv
        right_norm = right_vol_mm3 / icv
        total_norm = total_vol_mm3 / icv
    else:
        left_norm = right_norm = total_norm = np.nan

    feat = {
        "subject_id": subject_id,
        "scan_id": scan_id,
        "t1_path": t1_path,
        "mask_path": mask_path,
        "dx": dx,
        "icv": icv,
        "voxel_volume_mm3": voxel_volume,
        "left_hipp_voxels": left_voxels,
        "right_hipp_voxels": right_voxels,
        "left_hipp_vol_mm3": left_vol_mm3,
        "right_hipp_vol_mm3": right_vol_mm3,
        "total_hipp_vol_mm3": total_vol_mm3,
        "asymmetry_index": asymmetry_index,
        "left_hipp_vol_icv_norm": left_norm,
        "right_hipp_vol_icv_norm": right_norm,
        "total_hipp_vol_icv_norm": total_norm,
    }

    rows.append(feat)

df = pd.DataFrame(rows)

# 해마가 너무 작은(또는 0인) 케이스 제거용 임계값
MIN_VOXELS = 1000   # 필요하면 800, 1200 등으로 조정

is_valid = (
    (df["left_hipp_voxels"]  >= MIN_VOXELS) &
    (df["right_hipp_voxels"] >= MIN_VOXELS)
)

df_clean = df[is_valid].copy()
df_bad   = df[~is_valid].copy()

out_csv_clean = os.path.join(out_dir, "hippo_features_clean.csv")
out_csv_bad   = os.path.join(out_dir, "hippo_features_bad.csv")

df_clean.to_csv(out_csv_clean, index=False)
df_bad.to_csv(out_csv_bad, index=False)

print("\nSaved feature CSV (clean only)")
print(out_csv_clean)
print("유효 케이스 수:", len(df_clean), "/ 전체:", len(df))
print(df_clean.head())

print("\n이상치(제외된 케이스) CSV")
print(out_csv_bad)
print("제외된 케이스 수:", len(df_bad))

[경고] 메타데이터 파일 없음: D:\ADNI\metadata\adni_merged_icv.csv

[ Feature for: D:\ADNI\clean_masks_hippo\002_S_4213_20110902182731_hipp_clean.nii.gz ]
voxel sizes (mm): (1.2, 1.0, 1.0)
voxel volume (mm^3): 1.2
L voxels: 2715 R voxels: 2718
L vol: 3258.0001294612885 R vol: 3261.6001296043396

[ Feature for: D:\ADNI\clean_masks_hippo\002_S_4225_20110921100724_hipp_clean.nii.gz ]
voxel sizes (mm): (1.2, 1.0546875, 1.0546875)
voxel volume (mm^3): 1.3348389
L voxels: 2963 R voxels: 2879
L vol: 3955.1275634765625 R vol: 3843.0010986328125

[ Feature for: D:\ADNI\clean_masks_hippo\002_S_4262_20111005072430_hipp_clean.nii.gz ]
voxel sizes (mm): (1.2, 1.0, 1.0)
voxel volume (mm^3): 1.2
L voxels: 2288 R voxels: 2496
L vol: 2745.600109100342 R vol: 2995.2001190185547

[ Feature for: D:\ADNI\clean_masks_hippo\002_S_4264_20111005174145_hipp_clean.nii.gz ]
voxel sizes (mm): (1.2, 1.0, 1.0)
voxel volume (mm^3): 1.2
L voxels: 2336 R voxels: 2325
L vol: 2803.20011138916 R vol: 2790.0001108646393

[ Feature for